# Unit 1: Building the Symbolic Engine

So far, we have treated functions like black boxes: you put a number in, and a number comes out. We evaluated derivatives by looking at numbers getting infinitely close together.

But what if we want the computer to do algebra? What if we want it to know that the derivative of $x^2$ is exactly $2x$, not just an approximation? To do this, we have to teach Python how to read math as a structure, not just as a calculation.

We will do this by building an **Abstract Syntax Tree**.

Instead of writing `x + 3`, we will build objects that hold other objects, like `Add(Variable('x'), Constant(3))`. Let's build the base of our engine.


In [1]:
# The Base Class
# Every piece of math we build will inherit from this blueprint.
class Expression:
    def evaluate(self, context):
        # 'context' will be a dictionary, like {'x': 5}
        raise NotImplementedError("Each subclass must implement evaluate()")
        
    def __add__(self, other):
        # This allows us to use the standard '+' sign in Python later!
        return Add(self, other)
        
    def __mul__(self, other):
        return Multiply(self, other)



## 1. Constants and Variables

The foundational building blocks of any equation are numbers (Constants) and letters (Variables). Let's define their classes.

A `Constant` always evaluates to its own value. A `Variable` looks up its value in our `context` dictionary.


In [2]:
class Constant(Expression):
    def __init__(self, value):
        self.value = value
        
    def evaluate(self, context):
        return self.value
        
    def __str__(self):
        return str(self.value)

class Variable(Expression):
    def __init__(self, name):
        self.name = name
        
    def evaluate(self, context):
        if self.name in context:
            return context[self.name]
        else:
            raise ValueError(f"Value for variable '{self.name}' not provided in context.")
            
    def __str__(self):
        return self.name


## 2. Operations (Add and Multiply)

Now we need a way to combine our building blocks.

When we evaluate an `Add` object, it needs to evaluate its left side, evaluate its right side, and add the two numbers together. This is a recursive process!


In [3]:
class Add(Expression):
    def __init__(self, left, right):
        # We ensure everything is an Expression. 
        # If someone passes a raw integer, we convert it to a Constant.
        self.left = left if isinstance(left, Expression) else Constant(left)
        self.right = right if isinstance(right, Expression) else Constant(right)
        
    def evaluate(self, context):
        return self.left.evaluate(context) + self.right.evaluate(context)
        
    def __str__(self):
        return f"({self.left} + {self.right})"

class Multiply(Expression):
    def __init__(self, left, right):
        self.left = left if isinstance(left, Expression) else Constant(left)
        self.right = right if isinstance(right, Expression) else Constant(right)
        
    def evaluate(self, context):
        return self.left.evaluate(context) * self.right.evaluate(context)
        
    def __str__(self):
        return f"({self.left} * {self.right})"



## 3. Testing the Engine

Let's build the mathematical expression:


$$f(x) = 3x + 5$$

In our OOP framework, this becomes: `Add(Multiply(Constant(3), Variable('x')), Constant(5))`

Let's build it and evaluate it at $x = 4$.


In [4]:
# Build the pieces
x = Variable('x')
three = Constant(3)
five = Constant(5)

# Assemble the equation
# Because we defined __add__ and __mul__ in our base class, 
# we can use standard Python operators to combine our objects!
f = (three * x) + five

print("Our Symbolic Equation:", f)

# Evaluate it where x = 4
result = f.evaluate({'x': 4})
print("Evaluated at x=4:", result)


Our Symbolic Equation: ((3 * x) + 5)
Evaluated at x=4: 17
